# Search-7-MCTS-And-Beyond (C#) : Monte Carlo Tree Search et Extensions

**Navigation** : [<< Recherche adversariale (C#)](Search-6-AdversarialSearch-Csharp.ipynb) | [Index](../README.md) | [Version Python](Search-7-MCTS-And-Beyond.ipynb) | [Dancing Links >>](Search-8-DancingLinks.ipynb)

## Objectifs d'apprentissage

A la fin de ce notebook, vous saurez :
1. **Comprendre** les limites de Minimax et pourquoi MCTS a revolutionne les jeux
2. **Implementer** l'algorithme MCTS avec UCB1 en C#
3. **Comparer** MCTS vs Minimax sur differents types de jeux
4. **Explorer** les approches hybrides (AlphaGo, AlphaZero)
5. **Jouer** avec le parametre d'exploration et la convergence

### Prerequis
- Notebook Search-6-Csharp (AdversarialSearch, Minimax, Alpha-Beta)
- Bases de probabilites et statistiques
- Bases de C# : classes, generiques, LINQ

> **Jumeau .NET** du notebook Python [Search-7-MCTS-And-Beyond.ipynb](Search-7-MCTS-And-Beyond.ipynb). Meme pedagogie, implementation C# idiome. Les graphiques matplotlib sont remplaces par des tables texte (le kernel .NET Interactive fuiterait des chemins ScottPlot/SkiaSharp, verdict INTRINSIC #3436).

### Duree estimee : 90 minutes


In [1]:
// Cellule 1 : Environnement + interface du jeu (somme nulle)
using System.Diagnostics;
using System.Globalization;

// Contrat commun a tous les jeux a deux joueurs, somme nulle, tour a tour.
// TAction est fixe a int (TicTacToe = indice 0-8, Nim = 1-3, Connect-4 = colonne 0-6).
public interface IJeuSommeNulle<TEtat>
{
    TEtat EtatInitial();
    string Joueur(TEtat etat);                  // "MAX" ou "MIN"
    List<int> Actions(TEtat etat);
    TEtat Resultat(TEtat etat, int action);
    bool EstTerminal(TEtat etat);
    double Utilite(TEtat etat, string joueur);  // +1 victoire, -1 defaite, 0 nul
}

Console.WriteLine("Environnement pret pour MCTS (C# / .NET Interactive).");

The below script needs to be able to find the current output cell; this is an easy method to get it.

Environnement pret pour MCTS (C# / .NET Interactive).


## 1. Limites de Minimax

### Pourquoi Minimax ne suffit pas

L'algorithme Minimax avec Alpha-Beta fonctionne bien pour les jeux simples, mais rencontre des limites :

1. **Explosion combinatoire** : Aux echecs, meme avec Alpha-Beta, on ne peut explorer que 6-8 demi-coups en profondeur
2. **Fonction d'evaluation** : Necessite une expertise humaine pour concevoir une bonne heuristique
3. **Horizon effect** : Des evenements importants au-dela de la profondeur sont ignores

### La revolution AlphaGo (2016)

AlphaGo a battu Lee Sedol (champion du monde de Go) en utilisant :
- **MCTS** pour la recherche
- **Reseaux de neurones** pour l'evaluation (policy + value networks)

MCTS permet d'explorer intelligemment sans fonction d'evaluation experte !

> *Ancres savantes -- von Neumann (1928),Zur Theorie der Gesellschaftsspiele, Math. Annalen 100:295-320 (theoreme minimax) ; Kocsis & Szepesvari (2006), Bandit Based Monte-Carlo Planning, ECML, LNCS 4212:282-293 (UCT = MCTS + UCB1) ; Silver et al. (2016), Mastering the game of Go with deep neural networks and tree search, Nature 529:484-489 (AlphaGo) ; Silver et al. (2017), Mastering the game of Go without human knowledge, Nature 550:354-359 (AlphaGo Zero) ; Browne et al. (2012), A Survey of Monte Carlo Tree Search Methods, IEEE TCIAIG 4(1):1-43.*


## 2. L'Algorithme MCTS

### Principe

**Monte Carlo Tree Search** construit progressivement un arbre de recherche en :
1. **Selection** : Traverser l'arbre avec UCB1 pour equilibrer exploration/exploitation
2. **Expansion** : Ajouter un nouveau noeud a l'arbre
3. **Simulation** : Jouer une partie aleatoire jusqu'a la fin (rollout)
4. **Backpropagation** : Remonter le resultat dans l'arbre

### UCB1 (Upper Confidence Bound)

```
UCB1 = W/N + c * sqrt(ln(N_parent) / N)
```

- **W/N** : Taux de victoire (exploitation)
- **c * sqrt(...)** : Terme d'exploration (c = 1.41 = sqrt(2) typiquement)
- **N** : Nombre de visites du noeud ; **N_parent** : visites du parent


In [2]:
// Cellule 4 : Noeud de l'arbre MCTS (UCB1, selection, expansion)
public class NoeudMCTS<TEtat>
{
    public TEtat Etat { get; }
    public NoeudMCTS<TEtat>? Parent { get; }
    public int Action { get; }                         // coup qui a mene a cet etat (-1 = racine)
    public Dictionary<int, NoeudMCTS<TEtat>> Enfants { get; } = new();
    public int Visites { get; set; } = 0;
    public double Victoires { get; set; } = 0.0;
    public List<int>? ActionsNonExplorees { get; set; } // lazy a la premiere expansion

    public NoeudMCTS(TEtat etat, NoeudMCTS<TEtat>? parent = null, int action = -1)
    { Etat = etat; Parent = parent; Action = action; }

    // Convention UCT "moved-into" : Victoires/Visites est du point de vue du joueur qui a
    // DECIDE le coup vers ce noeud (voir Backpropagation dans MCTS<T>). Tous les enfants
    // d'un meme parent etant evalues du point de vue de ce parent, MeilleurEnfantUcb1
    // (max) selectionne bien le meilleur coup pour le joueur qui decide.

    // Score UCB1 : exploitation (W/N) + exploration (c * sqrt(ln(N_parent)/N))
    public double Ucb1(double c = 1.41)
    {
        if (Visites == 0) return double.PositiveInfinity;
        double exploitation = Victoires / Visites;
        double exploration = c * Math.Sqrt(Math.Log(Parent!.Visites) / Visites);
        return exploitation + exploration;
    }

    public NoeudMCTS<TEtat> MeilleurEnfantUcb1(double c = 1.41)
        => Enfants.Values.MaxBy(n => n.Ucb1(c))!;

    public NoeudMCTS<TEtat> MeilleurEnfantVisites()
        => Enfants.Values.MaxBy(n => n.Visites)!;

    public bool EstFullyExpanded(IJeuSommeNulle<TEtat> jeu)
    {
        ActionsNonExplorees ??= jeu.Actions(Etat);
        return ActionsNonExplorees.Count == 0;
    }

    public bool EstTerminal(IJeuSommeNulle<TEtat> jeu) => jeu.EstTerminal(Etat);
}

Console.WriteLine("Classe NoeudMCTS<T> definie (UCB1, selection, expansion).");

Classe NoeudMCTS<T> definie (UCB1, selection, expansion).



(5,28): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(10,21): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.

(12,50): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



Implementation de la classe MCTS avec selection UCB1, expansion, simulation (rollout aleatoire) et retropropagation.

In [3]:
// Cellule 6 : MCTS<T> -- selection (UCB1), expansion, simulation (rollout), backpropagation
public class MCTS<TEtat>
{
    private readonly IJeuSommeNulle<TEtat> _jeu;
    private readonly double _c;
    private readonly Random _rng;
    public Dictionary<string, int> Stats { get; } = new() { ["selections"]=0, ["expansions"]=0, ["simulations"]=0, ["backprops"]=0 };

    public MCTS(IJeuSommeNulle<TEtat> jeu, double c = 1.41, int? seed = null)
    { _jeu = jeu; _c = c; _rng = seed.HasValue ? new Random(seed.Value) : new Random(); }

    // Execute MCTS depuis l'etat donne ; retourne (meilleure action, valeur estimee).
    public (int Action, double Valeur) Recherche(TEtat etat, int iterations = 1000)
    {
        var racine = new NoeudMCTS<TEtat>(etat);
        for (int i = 0; i < iterations; i++)
        {
            var noeud = Selection(racine);
            double resultat = Simulation(noeud);
            Backpropagation(noeud, resultat);
        }
        var meilleur = racine.MeilleurEnfantVisites();
        double valeur = meilleur.Visites > 0 ? meilleur.Victoires / meilleur.Visites : 0;
        return (meilleur.Action, valeur);
    }

    // Variante qui expose aussi le nombre de visites du coup choisi (sert au benchmark de c).
    public (int Action, int Visites) RechercheAvecVisites(TEtat etat, int iterations)
    {
        var racine = new NoeudMCTS<TEtat>(etat);
        for (int i = 0; i < iterations; i++)
        {
            var noeud = Selection(racine);
            double resultat = Simulation(noeud);
            Backpropagation(noeud, resultat);
        }
        var meilleur = racine.MeilleurEnfantVisites();
        return (meilleur.Action, meilleur.Visites);
    }

    private NoeudMCTS<TEtat> Selection(NoeudMCTS<TEtat> noeud)
    {
        Stats["selections"]++;
        while (!noeud.EstTerminal(_jeu))
        {
            if (!noeud.EstFullyExpanded(_jeu)) return Expansion(noeud);
            noeud = noeud.MeilleurEnfantUcb1(_c);
        }
        return noeud;
    }

    private NoeudMCTS<TEtat> Expansion(NoeudMCTS<TEtat> noeud)
    {
        Stats["expansions"]++;
        noeud.ActionsNonExplorees ??= _jeu.Actions(noeud.Etat);
        int idx = noeud.ActionsNonExplorees.Count - 1;
        int action = noeud.ActionsNonExplorees[idx];     // pop (dernier element)
        noeud.ActionsNonExplorees.RemoveAt(idx);
        TEtat nouvelEtat = _jeu.Resultat(noeud.Etat, action);
        var enfant = new NoeudMCTS<TEtat>(nouvelEtat, noeud, action);
        noeud.Enfants[action] = enfant;
        return enfant;
    }

    // Rollout aleatoire depuis le noeud jusqu'a un etat terminal.
    private double Simulation(NoeudMCTS<TEtat> noeud)
    {
        Stats["simulations"]++;
        TEtat etat = noeud.Etat;
        // Le joueur MAX = celui qui devait jouer a l'etat parent (ou MAX a la racine).
        string joueurMaxOriginal = noeud.Parent != null ? _jeu.Joueur(noeud.Parent.Etat) : "MAX";
        while (!_jeu.EstTerminal(etat))
        {
            var actions = _jeu.Actions(etat);
            int action = actions[_rng.Next(actions.Count)];
            etat = _jeu.Resultat(etat, action);
        }
        return _jeu.Utilite(etat, joueurMaxOriginal);
    }

    private void Backpropagation(NoeudMCTS<TEtat>? noeud, double resultat)
    {
        Stats["backprops"]++;
        // Convention UCT "moved-into" : on stocke a chaque noeud la valeur du point de vue
        // du joueur QUI A DECIDE le coup vers ce noeud (le joueur de l'etat parent).
        // Ainsi tous les enfants d'un meme parent sont evalues du meme point de vue,
        // et max(UCB1) selectionne bien le meilleur coup pour le joueur qui decide.
        while (noeud != null)
        {
            noeud.Visites++;
            string decideur = noeud.Parent != null ? _jeu.Joueur(noeud.Parent.Etat) : _jeu.Joueur(noeud.Etat);
            if (decideur == "MAX") noeud.Victoires += resultat;
            else noeud.Victoires += -resultat;
            noeud = noeud.Parent;
        }
    }
}

Console.WriteLine("Classe MCTS<T> definie (selection, expansion, simulation, backpropagation).");

Classe MCTS<T> definie (selection, expansion, simulation, backpropagation).



(81,50): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



Maintenant que nous avons la structure de noeud, nous pouvons implementer l'algorithme **MCTS complet** qui utilise ces noeuds pour construire l'arbre de recherche.

## 3. Test sur Tic-Tac-Toe

Comparons MCTS avec Minimax sur le jeu de Morpion.


In [4]:
// Cellule 9 : Tic-Tac-Toe (implementation du jeu) + premier test MCTS
public sealed class EtatTTT
{
    public char[] Grille { get; }
    public char Joueur { get; }   // 'X' ou 'O' (celui qui doit jouer)
    public EtatTTT(char[] grille, char joueur) { Grille = grille; Joueur = joueur; }
}

public class TicTacToe : IJeuSommeNulle<EtatTTT>
{
    private static readonly int[][] Lignes = new[] {
        new[] {0,1,2}, new[] {3,4,5}, new[] {6,7,8},   // lignes
        new[] {0,3,6}, new[] {1,4,7}, new[] {2,5,8},   // colonnes
        new[] {0,4,8}, new[] {2,4,6}                    // diagonales
    };

    public EtatTTT EtatInitial() => new(Enumerable.Repeat(' ', 9).ToArray(), 'X');
    public string Joueur(EtatTTT e) => e.Joueur == 'X' ? "MAX" : "MIN";
    public List<int> Actions(EtatTTT e) => Enumerable.Range(0,9).Where(i => e.Grille[i] == ' ').ToList();
    public EtatTTT Resultat(EtatTTT e, int a)
    {
        char[] g = (char[])e.Grille.Clone();
        g[a] = e.Joueur;
        return new EtatTTT(g, e.Joueur == 'X' ? 'O' : 'X');
    }
    public bool EstTerminal(EtatTTT e)
    {
        foreach (var l in Lignes)
            if (e.Grille[l[0]] != ' ' && e.Grille[l[0]] == e.Grille[l[1]] && e.Grille[l[1]] == e.Grille[l[2]])
                return true;
        return !e.Grille.Contains(' ');
    }
    public double Utilite(EtatTTT e, string joueur)
    {
        foreach (var l in Lignes)
            if (e.Grille[l[0]] != ' ' && e.Grille[l[0]] == e.Grille[l[1]] && e.Grille[l[1]] == e.Grille[l[2]])
            {
                string gagnant = e.Grille[l[0]] == 'X' ? "MAX" : "MIN";
                return gagnant == joueur ? 1.0 : -1.0;
            }
        return 0.0;
    }
}

// Test MCTS sur Tic-Tac-Toe (etat initial vide)
var jeu = new TicTacToe();
var mcts = new MCTS<EtatTTT>(jeu, seed: 42);
var sw = Stopwatch.StartNew();
var (action, valeur) = mcts.Recherche(jeu.EtatInitial(), iterations: 1000);
sw.Stop();
Console.WriteLine($"MCTS (1000 iterations) : action={action}, valeur={valeur:F3}, temps={sw.Elapsed.TotalSeconds:F3}s");
Console.WriteLine($"Stats : selections={mcts.Stats["selections"]}, expansions={mcts.Stats["expansions"]}, simulations={mcts.Stats["simulations"]}, backprops={mcts.Stats["backprops"]}");

MCTS (1000 iterations) : action=1, valeur=0,041, temps=0,013s


Stats : selections=1000, expansions=1000, simulations=1000, backprops=1000


### Interpretation : Premiers resultats MCTS

La sortie ci-dessus montre la decision de MCTS apres 1000 iterations depuis la grille vide.

| Aspect | Signification |
|--------|---------------|
| **Action choisie** | Indice 0-8 ; un coin (0/2/6/8) ou le centre (4) sont les premiers coups strategiques au Morpion |
| **Valeur estimee** | Taux de victoire moyen des rollouts ; proche de 0 car le Morpion est un match nul entre joueurs competents |
| **Temps d'execution** | Tres rapide : MCTS n'explore qu'une partie de l'arbre (vs Minimax qui explore tout) |
| **Stats equilibrees** | selections = expansions = simulations = backprops = une iteration MCTS complete |

**Points cles** :
1. **Vitesse** : MCTS est nettement plus rapide que Minimax car il n'explore qu'une partie de l'arbre (comparaison chiffree en section 4).
2. **Valeur proche de 0** : Au Morpion, le resultat optimal est 0 (match nul), donc une valeur faible indique une bonne estimation.
3. **Action strategique** : Un coin ou le centre sont les meilleurs premiers coups au Morpion.
4. **Determinisme** : avec une graine fixee (`seed: 42`), la sortie est reproductible ; sans graine, elle varie legerement d'une execution a l'autre.


## 4. Comparaison MCTS vs Minimax

Comparons les deux approches sur differents criteres.


In [5]:
// Cellule 12 : Minimax (pour comparaison) + benchmark text-table Minimax vs MCTS
public static class RechercheAdversariale
{
    // Minimax exact (explore tout l'arbre) -- optimal mais exponentiel.
    public static (double Valeur, int? Action) Minimax<TEtat>(IJeuSommeNulle<TEtat> jeu, TEtat etat, string joueurMax = "MAX")
    {
        if (jeu.EstTerminal(etat)) return (jeu.Utilite(etat, joueurMax), null);
        var actions = jeu.Actions(etat);
        if (jeu.Joueur(etat) == joueurMax)
        {
            double bestV = double.NegativeInfinity; int? bestA = null;
            foreach (var a in actions)
            {
                var (v, _) = Minimax(jeu, jeu.Resultat(etat, a), joueurMax);
                if (v > bestV) { bestV = v; bestA = a; }
            }
            return (bestV, bestA);
        }
        else
        {
            double bestV = double.PositiveInfinity; int? bestA = null;
            foreach (var a in actions)
            {
                var (v, _) = Minimax(jeu, jeu.Resultat(etat, a), joueurMax);
                if (v < bestV) { bestV = v; bestA = a; }
            }
            return (bestV, bestA);
        }
    }
}

// Benchmark : Minimax vs MCTS a differents budgets d'iterations
Console.WriteLine($"{"Algorithme",-16}{"Temps (s)",12}{"Valeur",10}{"Action",8}");
Console.WriteLine(new string('-', 46));
var sw = Stopwatch.StartNew();
var (vMm, aMm) = RechercheAdversariale.Minimax(jeu, jeu.EtatInitial());
sw.Stop();
Console.WriteLine($"{"Minimax",-16}{sw.Elapsed.TotalSeconds,12:F4}{vMm,10:F2}{aMm,8}");
foreach (int nIter in new[] { 100, 500, 1000, 5000 })
{
    var m = new MCTS<EtatTTT>(jeu, seed: 42);
    sw.Restart();
    var (action, valeur) = m.Recherche(jeu.EtatInitial(), nIter);
    sw.Stop();
    Console.WriteLine($"{"MCTS ("+nIter+")",-16}{sw.Elapsed.TotalSeconds,12:F4}{valeur,10:F2}{action,8}");
}

Algorithme         Temps (s)    Valeur  Action


----------------------------------------------


Minimax               0,3076      0,00       0


MCTS (100)            0,0005      0,27       8


MCTS (500)            0,0027      0,12       1


MCTS (1000)           0,0052      0,04       1


MCTS (5000)           0,0264      0,27       1


### Interpretation : Comparaison MCTS vs Minimax

La table ci-dessus compare Minimax (exploration complete, exact) a MCTS a 4 budgets d'iterations.

**Points cles** (a lire sur la sortie reelle ci-dessus) :
1. **Avantage vitesse critique** : MCTS meme a faible budget est considerablement plus rapide que Minimax, qui explore tout l'arbre du Morpion (~26 000 positionslegales avant elagage terminal).
2. **Convergence progressive** : plus d'iterations rapprochent la valeur MCTS de 0 (la valeur optimale donnee par Minimax).
3. **Stabilite tardive** : a 5000 iterations, la valeur estimee MCTS est tres proche de l'optimal (0).
4. **Variabilite des actions** : les actions choisies varient selon le budget, montrant que le nombre d'iterations influence la stabilite du choix.

> **Note technique** : La valeur de Minimax (0) est exacte car l'algorithme explore tout l'arbre de jeu du Morpion. La valeur MCTS est une estimation probabiliste qui converge vers 0 avec plus d'iterations.


### Analyse des resultats

| Critere | Minimax | MCTS |
|---------|---------|------|
| **Garantie d'optimalite** | Oui (si arbre complet) | Non (probabiliste) |
| **Fonction d'evaluation** | Necessaire (pour la profondeur) | Non necessaire |
| **Complexite** | O(b^d) | O(n * d) ou n = iterations |
| **Parallellisable** | Difficile | Tres facile |
| **Jeux a grand facteur de branchement** | Impraticable | Adapte |

### Quand utiliser MCTS ?

- Jeux avec grand espace d'etat (Go, Hex)
- Jeux sans bonne fonction d'evaluation
- Situations avec contrainte de temps variable
- Jeux avec hasard (backgammon, poker)


## 5. OpenSpiel -- Framework de Jeux

### Presentation

**OpenSpiel** est un framework open-source de DeepMind pour la recherche en IA sur les jeux.

### Caracteristiques

- **40+ jeux** : Echecs, Go, Hex, Poker, Hanabi, etc.
- **Multi-agent** : jeux a N joueurs
- **Information imparfaite** : Poker, Hanabi
- **Algos inclus** : MCTS, AlphaZero, CFR, etc.

> **Note .NET** : OpenSpiel est une librairie C++/Python (installation `pip install open-spiel`, Linux/WSL). Il n'existe pas d'equivalent .NET direct ; en C# on reimplemente la logique de jeu (comme nous le faisons ici avec `IJeuSommeNulle<T>`) ou on consomme des moteurs natifs via P/Invoke. La cellule suivante montre l'equivalent conceptuel de l'API OpenSpiel en C#.


In [6]:
// Cellule 16 : Equivalent conceptuel de l'API OpenSpiel en C#
// OpenSpiel (pyspiel) est C++/Python uniquement. En .NET on reimplemente le jeu
// derriere une interface normalisee (c'est exactement IJeuSommeNulle<TEtat> ci-dessus).
// On reutilise donc notre TicTacToe + MCTS pour montrer le meme usage qu'OpenSpiel.

var openspielEquivalent = new TicTacToe();
var bot = new MCTS<EtatTTT>(openspielEquivalent, c: 1.41, seed: 42);
var etat0 = openspielEquivalent.EtatInitial();
var (actionOS, _) = bot.Recherche(etat0, iterations: 1000);
Console.WriteLine($"Jeu charge        : tic_tac_toe()  (via TicTacToe : IJeuSommeNulle<EtatTTT>)");
Console.WriteLine($"Etat initial      : grille vide, X a jouer");
Console.WriteLine($"Action MCTSBot    : {actionOS}  (1000 simulations, c=sqrt(2))");
Console.WriteLine($"\nOpenSpiel .NET : pas de port officiel. L'interface IJeuSommeNulle<T> normalise");
Console.WriteLine($"  la meme separation jeu / algorithme qu'OpenSpiel (state, actions, resultat, terminal, utilite).");

Jeu charge        : tic_tac_toe()  (via TicTacToe : IJeuSommeNulle<EtatTTT>)


Etat initial      : grille vide, X a jouer


Action MCTSBot    : 1  (1000 simulations, c=sqrt(2))



OpenSpiel .NET : pas de port officiel. L'interface IJeuSommeNulle<T> normalise


  la meme separation jeu / algorithme qu'OpenSpiel (state, actions, resultat, terminal, utilite).


### Interpretation : OpenSpiel et la normalisation des jeux

**Points cles** :
1. **Interface normalisee** : `IJeuSommeNulle<T>` joue le role de l'API `pyspiel` -- n'importe quel jeu a deux joueurs l'implemente et beneficie immediatement de MCTS, Minimax, etc.
2. **Pas de port .NET officiel** : OpenSpiel est C++/Python ; en C# on reimplemente la logique (comme ici) ou on invoque un moteur natif.
3. **Coherence avec MCTS manuel** : le bot joue un coup fort (coin/centre), comme le `MCTSBot` d'OpenSpiel.
4. **Separation jeu / algorithme** : c'est le pattern cle -- ajouter un nouveau jeu (Connect-4, Hex...) se reduit a implementer 6 methodes.


## 6. AlphaGo et AlphaZero

### Architecture AlphaGo (2016)

AlphaGo combine MCTS avec des reseaux de neurones :
1. **Policy Network** : predit la probabilite de chaque coup
2. **Value Network** : evalue la position
3. **MCTS** : guide la recherche avec les predictions des reseaux

### AlphaZero (2017)

AlphaZero simplifie et generalise l'approche :
- **Auto-apprentissage** : pas de donnees humaines
- **Reseau unique** : policy + value ensemble
- **Universel** : fonctionne pour Go, Echecs, Shogi

### Principe de l'apprentissage

```
1. Initialiser le reseau aleatoirement
2. Jouer des parties contre soi-meme avec MCTS
3. Entrainer le reseau sur les positions et resultats
4. Repeter jusqu'a convergence
```


In [7]:
// Cellule 19 : Sketch conceptuel d'un AlphaZero simplifie (C#)
// En pratique, policy_value_fn serait un reseau de neurones (PyTorch/TensorFlow/.NET).
// Ici : une fonction factice qui simule (probas, valeur) pour illustrer l'integration MCTS.

// Signature d'une evaluation reseau : (probas par action, valeur de la position)
public delegate (Dictionary<int,double> Probs, double Value) PolicyValueFn<TEtat>(TEtat etat);

public class AlphaZeroMCTS<TEtat>
{
    private readonly IJeuSommeNulle<TEtat> _jeu;
    private readonly PolicyValueFn<TEtat> _policyValue;
    private readonly double _c;
    public AlphaZeroMCTS(IJeuSommeNulle<TEtat> jeu, PolicyValueFn<TEtat> policyValue, double c = 1.41)
    { _jeu = jeu; _policyValue = policyValue; _c = c; }

    // MCTS guide par le reseau : l'expansion utilise les probas policy,
    // et la simulation est remplacee par l'evaluation value (pas de rollout aleatoire).
    public NoeudMCTS<TEtat> Recherche(TEtat etat, int iterations = 100)
    {
        var racine = new NoeudMCTS<TEtat>(etat);
        for (int i = 0; i < iterations; i++)
        {
            var noeud = racine;
            // Selection (comme MCTS classique, via UCB1)
            while (noeud.Enfants.Count > 0 && noeud.EstFullyExpanded(_jeu) && !_jeu.EstTerminal(noeud.Etat))
                noeud = noeud.MeilleurEnfantUcb1(_c);
            // Evaluation par le reseau
            double value;
            if (_jeu.EstTerminal(noeud.Etat))
                value = _jeu.Utilite(noeud.Etat, "MAX");
            else
            {
                var (probs, v) = _policyValue(noeud.Etat);
                value = v;
                // Expansion avec les probabilites du policy network
                foreach (var a in _jeu.Actions(noeud.Etat))
                    noeud.Enfants[a] = new NoeudMCTS<TEtat>(_jeu.Resultat(noeud.Etat, a), noeud, a);
            }
            // Backpropagation (comme MCTS classique ; Victoires est un setter public)
            while (noeud != null)
            {
                noeud.Visites++;
                if (_jeu.Joueur(noeud.Etat) == "MAX") noeud.Victoires += value;
                else noeud.Victoires += -value;
                noeud = noeud.Parent;
            }
        }
        return racine;
    }
}

Console.WriteLine("Sketch AlphaZeroMCTS<T> defini (policy network + value network, pas de rollout aleatoire).");

Sketch AlphaZeroMCTS<T> defini (policy network + value network, pas de rollout aleatoire).


### Interpretation : Concept AlphaZero et integration reseau/MCTS

| Aspect | Implementation | Signification |
|--------|----------------|---------------|
| **Policy/Value** | `PolicyValueFn<T>` delegate | Simule un reseau (a remplacer par PyTorch/TensorFlow/.NET) |
| **Integration MCTS** | Selection + Expansion guidee | MCTS utilise les probabilites du policy network |
| **Backpropagation** | Valeur reseau (pas rollout) | Plus efficace que les rollouts aleatoires |
| **Architecture** | Reseau unique (policy+value) | Simplification d'AlphaZero vs AlphaGo (2016) |

**Points cles** :
1. **Remplacement du rollout** : AlphaZero remplace les simulations aleatoires par l'evaluation directe du value network.
2. **Policy network** : guide l'expansion en donnant des probabilites pour chaque action.
3. **Auto-apprentissage** : le reseau s'entraine sur les parties generees par MCTS (auto-play sans donnees humaines).
4. **Generalisation** : la meme architecture fonctionne pour Go, Echecs, Shogi.


## 7. Extensions Avancees

### RAVE (Rapid Action Value Estimation)

RAVE utilise l'heuristique "all moves as first" pour accelerer l'apprentissage : un coup est evalue meme s'il est joue plus tard dans la partie. Acceleration significative dans les premiers stades.

### UCT (UCB applied to Trees)

UCT est le nom original de l'algorithme MCTS avec UCB1. Variantes :
- **UCB1-Tuned** : ajuste le parametre d'exploration
- **UCB-V** : utilise la variance des recompenses

### Parallelisation

MCTS se parallellise facilement :
- **Leaf parallelisation** : simulations en parallele
- **Root parallelisation** : plusieurs arbres en parallele
- **Tree parallelisation** : acces concurrent a l'arbre


In [8]:
// Cellule 22 : Root parallelization en C# (Parallel.ForEach / LINQ)
// Contrairement a multiprocessing.Pool (Python), le TPL (.NET) fonctionne dans un notebook.
public static class MCTSParallel
{
    // Root parallelization : nWorkers arbres MCTS independants, vote majoritaire.
    public static int Recherche<TEtat>(IJeuSommeNulle<TEtat> jeu, TEtat etat, int totalIterations, int nWorkers, int seed = 42)
    {
        int perWorker = Math.Max(1, totalIterations / nWorkers);
        var actions = new int[nWorkers];
        Parallel.For(0, nWorkers, w =>
        {
            var m = new MCTS<TEtat>(jeu, seed: seed + w);
            actions[w] = m.Recherche(etat, perWorker).Action;
        });
        // Vote majoritaire
        return actions.GroupBy(a => a).OrderByDescending(g => g.Count()).First().Key;
    }
}

var jeuP = new TicTacToe();
var sw = Stopwatch.StartNew();
int coupVote = MCTSParallel.Recherche(jeuP, jeuP.EtatInitial(), totalIterations: 1000, nWorkers: 4, seed: 42);
sw.Stop();
Console.WriteLine($"Root parallelization (4 workers x 250 iter) -> action = {coupVote}, temps = {sw.Elapsed.TotalSeconds:F3}s");
Console.WriteLine("Chaque worker construit son propre arbre ; le vote majoritaire reduit la variance.");

Root parallelization (4 workers x 250 iter) -> action = 1, temps = 0,006s


Chaque worker construit son propre arbre ; le vote majoritaire reduit la variance.


### Interpretation : Parallelisation de MCTS

| Aspect | Implementation | Signification |
|--------|----------------|---------------|
| **Type** | Root parallelization | Plusieurs arbres MCTS independants en parallele |
| **Workers** | 4 (configurable) | Chaque worker execute un MCTS complet |
| **Vote** | `GroupBy().OrderByDescending(Count)` | L'action la plus choisie parmi les workers gagne |
| **API .NET** | `Parallel.For` / TPL | Fonctionne dans un notebook (contrairement au `multiprocessing` Python) |

**Points cles** :
1. **Root parallelization** : chaque worker construit son propre arbre MCTS independant, puis on vote.
2. **Scalabilite** : avec 4 coeurs, on fait ~4x plus d'iterations dans le meme temps.
3. **Vote majoritaire** : reduit la variance en combinant plusieurs estimations.
4. **Alternative** : tree parallelization (acces concurrent a un arbre partage avec lock) -- plus complexe mais meilleure convergence.


## Exemples et Exercices

Les deux premiers items sont des **exemples completement resolus** qui servent de modeles. Les exercices 2 a 5 sont des stubs a completer (C# `// TODO`, sans lancer d'exception -- le notebook doit s'executer de bout en bout).

### Exemple resolu 1 : Analyse de convergence MCTS
Mesurez comment la qualite des decisions MCTS evolue avec le nombre d'iterations.

### Exemple resolu 2 : MCTS sur le jeu de Nim
Verifiez que MCTS decouvre la strategie optimale connue du Nim (laisser un multiple de 4).

### Exercice 2 : Rollout intelligent
Implementez un rollout guide par heuristiques au lieu d'un choix purement aleatoire.

### Exercice 3 : MCTS vs Alpha-Beta sur Connect Four
Comparez les deux algorithmes sur le jeu de Puissance 4.

### Exercice 4 : Extension RAVE
Implementez l'extension RAVE pour accelerer la convergence.

### Exercice 5 : Time management
Implementez une gestion du temps adaptive pour MCTS.


### Exemple resolu : Analyse de convergence MCTS

La convergence est une propriete fondamentale de MCTS : plus on augmente le nombre d'iterations, plus la decision se rapproche de l'optimal. L'exemple ci-dessous mesure cette convergence sur TicTacToe en comparant l'action choisie par MCTS a l'action optimale Minimax (verite terrain). La table montre le taux de choix optimal, la valeur estimee moyenne et le temps moyen, pour des budgets croissants.


In [9]:
// Exemple resolu : Analyse de convergence MCTS (table texte au lieu de matplotlib)
// Pour chaque budget d'iterations, on repete N fois et on mesure le taux de choix
// de l'action optimale (ref Minimax), la valeur estimee moyenne et le temps moyen.
static List<(int Iter, double TauxOpt, double ValeurMoy, double TempsMoy)> AnalyserConvergence<TEtat>(
    IJeuSommeNulle<TEtat> jeu, TEtat etat, IEnumerable<int> iterList, int repetitions = 20, int seed = 42)
{
    var (_, actionOptNull) = RechercheAdversariale.Minimax(jeu, etat);
    int actionOpt = actionOptNull ?? jeu.Actions(etat)[0];
    var rng = new Random(seed);
    var res = new List<(int, double, double, double)>();
    foreach (int nIter in iterList)
    {
        int optCount = 0;
        double sv = 0, st = 0;
        for (int r = 0; r < repetitions; r++)
        {
            var mcts = new MCTS<TEtat>(jeu, seed: rng.Next());
            var sw = Stopwatch.StartNew();
            var (action, valeur) = mcts.Recherche(etat, nIter);
            sw.Stop();
            if (action == actionOpt) optCount++;
            sv += valeur; st += sw.Elapsed.TotalSeconds;
        }
        res.Add((nIter, optCount * 100.0 / repetitions, sv / repetitions, st / repetitions));
    }
    return res;
}

var iterList = new[] { 10, 50, 100, 500, 1000, 2000 };
var conv = AnalyserConvergence(jeu, jeu.EtatInitial(), iterList, repetitions: 20);
Console.WriteLine($"{"Iterations",12}{"Taux optimal (%)",18}{"Valeur moyenne",16}{"Temps moyen (s)",16}");
Console.WriteLine(new string('-', 62));
foreach (var (it, taux, vmoy, tmoy) in conv)
    Console.WriteLine($"{it,12}{taux,18:F1}{vmoy,16:F3}{tmoy,16:F4}");
Console.WriteLine("\nRemarque : depuis la grille vide, plusieurs premiers coups du Morpion sont");
Console.WriteLine("tous optimaux (valeur 0). Le taux compare a UNE action de reference (Minimax).");

  Iterations  Taux optimal (%)  Valeur moyenne Temps moyen (s)


--------------------------------------------------------------


          10               0,0           0,100          0,0000


          50               5,0           0,179          0,0002


         100              15,0           0,294          0,0005


         500               5,0           0,138          0,0028


        1000               5,0           0,040          0,0048


        2000              10,0           0,024          0,0109



Remarque : depuis la grille vide, plusieurs premiers coups du Morpion sont


tous optimaux (valeur 0). Le taux compare a UNE action de reference (Minimax).


### Exemple resolu : MCTS sur le jeu de Nim

Le jeu de **Nim** est un cas ideal pour comprendre MCTS : l'espace d'etats est petit et la strategie optimale est connue mathematiquement (laisser un multiple de 4 allumettes a l'adversaire). L'exemple ci-dessous montre la **convergence** de MCTS vers cette strategie en fonction du budget d'iterations, puis compare ses performances contre un joueur optimal. On observe que MCTS decouvre l'action optimale (prendre 3 depuis n=15) des 1000 iterations, avec une valeur estimee qui converge vers la victoire (0,64 -> 0,96).


In [10]:
// Exemple resolu : MCTS sur le jeu de Nim
public sealed class EtatNim { public int Restantes; public char Joueur; public EtatNim(int r, char j){Restantes=r;Joueur=j;} }

public class NimGame : IJeuSommeNulle<EtatNim>
{
    private readonly int _n0;
    public NimGame(int nAllumettes = 15) => _n0 = nAllumettes;
    public EtatNim EtatInitial() => new(_n0, 'X');
    public string Joueur(EtatNim e) => e.Joueur == 'X' ? "MAX" : "MIN";
    public List<int> Actions(EtatNim e) => new[] { 1, 2, 3 }.Where(k => k <= e.Restantes).ToList();
    public EtatNim Resultat(EtatNim e, int a) => new(e.Restantes - a, e.Joueur == 'X' ? 'O' : 'X');
    public bool EstTerminal(EtatNim e) => e.Restantes == 0;
    public double Utilite(EtatNim e, string joueur)
    {
        // e = (0, joueur qui doit jouer) -> il ne peut pas jouer -> l'autre a pris la derniere -> l'autre GAGNE
        string gagnant = e.Joueur == 'O' ? "MAX" : "MIN";
        return gagnant == joueur ? 1.0 : -1.0;
    }
}

static int StrategieOptimaleNim(EtatNim etat)
{
    int reste = etat.Restantes % 4;
    return reste > 0 ? reste : 1;   // laisser un multiple de 4 a l'adversaire
}

var jeuNim = new NimGame(15);
// Budget eleve pour l'action initiale : MCTS doit explorer assez pour decouvrir la parite (n%%4).
Console.WriteLine("--- Action initiale : MCTS a budgets croissants ---");
int actionOptNim = StrategieOptimaleNim(jeuNim.EtatInitial());
Console.WriteLine($"Nim(15) - Strategie optimale : prendre {actionOptNim} (laisser un multiple de 4)\n");
foreach (int nIter in new[] { 1000, 5000, 10000 })
{
    var (a, v) = new MCTS<EtatNim>(jeuNim, seed: 42).Recherche(jeuNim.EtatInitial(), nIter);
    Console.WriteLine($"  MCTS({nIter,5} iter) : prendre {a}, valeur={v:F3}  {(a == actionOptNim ? "<< OPTIMAL" : "")}");
}
Console.WriteLine();
// Tournoi : MCTS(500) vs strategie optimale, 50 parties (MCTS joue 1er puis 2nd)
int victoiresMcts = 0, nulles = 0, nParties = 50;
for (int i = 0; i < nParties; i++)
{
    var etat = jeuNim.EtatInitial();
    string mctsJoueur = i % 2 == 0 ? "MAX" : "MIN";
    while (!jeuNim.EstTerminal(etat))
    {
        int a;
        if (jeuNim.Joueur(etat) == mctsJoueur)
            a = new MCTS<EtatNim>(jeuNim, seed: 1000 + i).Recherche(etat, 500).Action;
        else
            a = StrategieOptimaleNim(etat);
        etat = jeuNim.Resultat(etat, a);
    }
    double u = jeuNim.Utilite(etat, mctsJoueur);
    if (u > 0) victoiresMcts++; else if (u == 0) nulles++;
}
Console.WriteLine($"\nTournoi MCTS(500 iter) vs Strategie optimale ({nParties} parties) :");
Console.WriteLine($"  Victoires MCTS              : {victoiresMcts}/{nParties}");
Console.WriteLine($"  Nulles                      : {nulles}/{nParties}");
Console.WriteLine($"  Victoires strategie optimale: {nParties - victoiresMcts - nulles}/{nParties}");

--- Action initiale : MCTS a budgets croissants ---


Nim(15) - Strategie optimale : prendre 3 (laisser un multiple de 4)



  MCTS( 1000 iter) : prendre 3, valeur=0,642  << OPTIMAL


  MCTS( 5000 iter) : prendre 3, valeur=0,929  << OPTIMAL


  MCTS(10000 iter) : prendre 3, valeur=0,963  << OPTIMAL



Tournoi MCTS(500 iter) vs Strategie optimale (50 parties) :


  Victoires MCTS              : 6/50


  Nulles                      : 0/50


  Victoires strategie optimale: 44/50


### Exercice 2 : Rollout intelligent

Implementez un rollout guide par heuristiques au lieu d'un choix purement aleatoire. Evaluez les actions candidates avec une fonction heuristique et choisissez l'action avec le meilleur score (avec un peu d'aleatoire). Pour TicTacToe, privilegiez le centre et les coins.


In [11]:
// Exercice 2 : Rollout intelligent (stub a completer)
// Indice : derivez de MCTS<T> et surchargez la methode de selection d'action pendant
// le rollout. Pour TicTacToe, une heuristique simple = score(centre=3, coin=2, bord=1).
public class MCTSRolloutHeuristique<TEtat> : MCTS<TEtat>
{
    public MCTSRolloutHeuristique(IJeuSommeNulle<TEtat> jeu, double c = 1.41, int? seed = null)
        : base(jeu, c, seed) { }

    // TODO etudiant : remplacer le rollout aleatoire par un rollout guide.
    // 1. Definir une fonction heuristique Heuristique(etat) -> score par action.
    //    Ex TicTacToe : centre (action 4) = 3, coins (0,2,6,8) = 2, bords = 1.
    // 2. Avec probabilite epsilon, jouer aleatoire (exploration) ; sinon jouer le meilleur score.
    // 3. Mesurer la convergence plus rapide vs rollout purement aleatoire.
    public double HeuristiqueExample(TEtat etat, int action)
    {
        // TODO etudiant : implementer l'heuristique selon le jeu.
        return 0.0;  // placeholder : retour neutre (a completer)
    }
}
Console.WriteLine("Exercice 2 : stub pret -- derivez MCTS<T> et implementez le rolloutheuristique.");

Exercice 2 : stub pret -- derivez MCTS<T> et implementez le rolloutheuristique.


### Exercice 3 : MCTS vs Alpha-Beta sur Connect Four

Comparez les deux algorithmes sur le jeu de Puissance 4. Implementez Connect Four (ou reutilisez du notebook Search-6), faites jouer MCTS vs Alpha-Beta sur plusieurs parties et analysez : taux de victoire, temps, qualite des coups. Alpha-Beta devrait etre plus fort avec une bonne profondeur.


In [12]:
// Exercice 3 : MCTS vs Alpha-Beta sur Connect Four (stub a completer)
// Indice : Connect Four = grille 6x7, gravite (le jeton tombe en bas de la colonne).
// 1. Implementer une classe ConnectFour : IJeuSommeNulle<EtatC4>.
//    - EtatC4 = { char[42] Grille, char Joueur }
//    - Actions = colonnes non pleines (0-6)
//    - Resultat : trouver la 1ere case vide en partant du bas dans la colonne
//    - EstTerminal : alignement de 4 (H, V, 2 diagonales) ou grille pleine
// 2. Ajouter Alpha-Beta (Minimax + elagage) -- reutilisable depuis Search-6-Csharp.
// 3. Tournoi MCTS(iter) vs AlphaBeta(prof) sur N parties.
public sealed class EtatC4 { /* TODO etudiant */ }
public class ConnectFour : IJeuSommeNulle<EtatC4>
{
    public EtatC4 EtatInitial() => default!;   // TODO etudiant
    public string Joueur(EtatC4 e) => "MAX";    // TODO etudiant
    public List<int> Actions(EtatC4 e) => new();// TODO etudiant
    public EtatC4 Resultat(EtatC4 e, int a) => default!;  // TODO etudiant
    public bool EstTerminal(EtatC4 e) => true;  // TODO etudiant
    public double Utilite(EtatC4 e, string j) => 0.0;     // TODO etudiant
}
Console.WriteLine("Exercice 3 : stub ConnectFour pret -- implementez les 6 methodes de l'interface.");

Exercice 3 : stub ConnectFour pret -- implementez les 6 methodes de l'interface.


### Exercice 4 : Extension RAVE

Implementez l'extension RAVE (Rapid Action Value Estimation) pour accelerer la convergence de MCTS. RAVE utilise l'heuristique "all moves as first" : un coup est evalue meme s'il est joue plus tard dans la partie. La formule est `Q_RAVE = (1-beta) * Q_MCTS + beta * Q_RAVE`. Modifiez la classe `NoeudMCTS<T>` pour stocker les stats RAVE.


In [13]:
// Exercice 4 : Extension RAVE (stub a completer)
// Indice : ajouter a NoeudMCTS<T> deux compteurs par action globale (pas seulement par noeud).
public class NoeudRAVE<TEtat> : NoeudMCTS<TEtat>
{
    public Dictionary<int, int> RaveVisites = new();   // TODO etudiant : remplir pendant la backprop
    public Dictionary<int, double> RaveVictoires = new();
    public NoeudRAVE(TEtat etat, NoeudMCTS<TEtat>? parent = null, int action = -1)
        : base(etat, parent, action) { }

    // TODO etudiant : surcharger Ucb1 pour melanger Q_MCTS et Q_RAVE.
    // Q_RAVE = (1-beta) * (Victoires/Visites) + beta * (RaveVictoires[a]/RaveVisites[a])
    // avec beta decroissant avec le nombre de visites.
    public double Ucb1Rave(double c = 1.41, double beta = 0.5)
    {
        // TODO etudiant : implementer la formule RAVE
        return Ucb1(c);   // placeholder = UCB1 classique (a completer)
    }
}
Console.WriteLine("Exercice 4 : stub NoeudRAVE<T> pret -- ajoutez les stats RAVE et la formule beta.");

Exercice 4 : stub NoeudRAVE<T> pret -- ajoutez les stats RAVE et la formule beta.



(7,50): warning CS8632: L'annotation pour les types référence Nullable doit être utilisée uniquement dans le code au sein d'un contexte d'annotations '#nullable'.



### Exercice 5 : Time management

Implementez une gestion du temps adaptive pour MCTS. Allouez plus de temps aux positions critiques, utilisez moins de temps quand le coup est evident, et detectez les positions "faciles" (victoire probable). Utilisez la variance des evaluations pour detecter l'incertitude.


In [14]:
// Exercice 5 : Time management adaptive (stub a completer)
// Indice : au lieu d'un budget fixe d'iterations, allouer un budget variable selon :
//  - la variance des valeurs estimees des enfants (haute variance = incertain = + d'iterations)
//  - la presence d'un coup ecrasant (un enfant >> les autres = coup evident = - d'iterations)
public static class TimeManagement
{
    // TODO etudiant : retourner un nombre d'iterations adapte a la position.
    public static int BudgetAdaptatif<TEtat>(IJeuSommeNulle<TEtat> jeu, TEtat etat, int budgetMax)
    {
        // 1. Lancer un pre-MCTS court (budgetMax/10 iter) pour estimer la variance.
        // 2. Si variance faible (coup evident) -> retourner budgetMax/4.
        // 3. Si variance elevee (position critique) -> retourner budgetMax.
        return budgetMax;   // placeholder = budget fixe (a completer)
    }
}
Console.WriteLine("Exercice 5 : stub TimeManagement pret -- implementez le budget adaptatif.");

Exercice 5 : stub TimeManagement pret -- implementez le budget adaptatif.


### Exemple : Parametre d'exploration c

L'exemple ci-dessous benchmark differentes valeurs du parametre d'exploration `c` de MCTS. On joue des parties MCTS contre un joueur aleatoire, puis on compare les taux de victoire, temps d'execution et visites moyennes pour `c = 0.5, 1.0, 1.41, 2.0`.

Observez comment la valeur de `c` influence le compromis entre exploration et exploitation.


In [15]:
// Exemple : Benchmark du parametre d'exploration c
// Pour chaque c, on joue N parties MCTS(c) contre un joueur aleatoire, et on mesure
// le taux de victoire, le temps moyen et les visites moyennes du coup choisi.
static (double TauxVictoire, double TempsMoy, double VisitesMoy) JouerPartieMctsVsRandom<TEtat>(
    IJeuSommeNulle<TEtat> jeu, MCTS<TEtat> mcts, int iterations, int seed)
{
    var rng = new Random(seed);
    var etat = jeu.EtatInitial();
    var visites = new List<int>();
    bool tourMcts = true;
    while (!jeu.EstTerminal(etat))
    {
        int a;
        if (tourMcts)
        {
            var (act, vis) = mcts.RechercheAvecVisites(etat, iterations);
            a = act; visites.Add(vis);
        }
        else
        {
            var acts = jeu.Actions(etat);
            a = acts[rng.Next(acts.Count)];
        }
        etat = jeu.Resultat(etat, a);
        tourMcts = !tourMcts;
    }
    double res = jeu.Utilite(etat, "MAX");
    double visMoy = visites.Count > 0 ? visites.Average() : 0.0;
    return (res, 0.0, visMoy);
}

Console.WriteLine($"{"c",6}{"Taux victoire (%)",20}{"Temps moyen (s)",18}{"Visites moyennes",18}");
Console.WriteLine(new string('-', 62));
foreach (double c in new[] { 0.5, 1.0, 1.41, 2.0 })
{
    int victoires = 0; double sommeTemps = 0, sommeVisites = 0;
    int parties = 12;
    for (int p = 0; p < parties; p++)
    {
        var mcts = new MCTS<EtatTTT>(jeu, c: c, seed: 500 + p);
        var sw = Stopwatch.StartNew();
        var (res, _, visMoy) = JouerPartieMctsVsRandom(jeu, mcts, iterations: 1000, seed: 900 + p);
        sw.Stop();
        if (res > 0) victoires++;
        sommeTemps += sw.Elapsed.TotalSeconds;
        sommeVisites += visMoy;
    }
    Console.WriteLine($"{c,6}{victoires * 100.0 / parties,20:F1}{sommeTemps / parties,18:F4}{sommeVisites / parties,18:F1}");
}

     c   Taux victoire (%)   Temps moyen (s)  Visites moyennes


--------------------------------------------------------------


   0,5                83,3            0,0151             831,4


     1                83,3            0,0128             704,4


  1,41               100,0            0,0124             556,4


     2                91,7            0,0122             349,3


---

### Exercice : MCTS sur le jeu de Connect-4

Implementez le jeu de **Connect-4** (Puissance 4) et testez l'algorithme MCTS dessus (voir stub Exercice 3).

**Regles du Connect-4** :
- Grille de 6 lignes x 7 colonnes
- Deux joueurs (X et O) placent tour a tour un jeton dans une colonne
- Le jeton tombe dans la case vide la plus basse de la colonne
- Le premier joueur a aligner 4 jetons (H, V, ou diagonale) gagne
- Si la grille est pleine sans alignement, c'est un match nul

**Contraintes** :
- L'etat sera represente par une classe `EtatC4` (grille `char[42]`, joueur courant)
- Les actions sont les indices de colonnes valides (0 a 6)
- Le facteur de branchement est 7 (vs 9 au Tic-Tac-Toe), l'arbre est beaucoup plus grand

**Indices** :
- Pour `Resultat`, les jetons tombent : trouver la premiere case vide dans la colonne en partant du bas
- Pour `EstTerminal`, verifiez les 4 directions : horizontal, vertical, et les deux diagonales
- Le nombre de positions au Connect-4 est d'environ 4.5 trillions, MCTS est donc particulierement pertinent


In [16]:
// Exercice : MCTS sur Connect-4 (la classe ConnectFour est declaree en Exercice 3).
// Une fois les 6 methodes implementees, decommentez le tournoi ci-dessous.
// var jeuC4 = new ConnectFour();
// var mctsC4 = new MCTS<EtatC4>(jeuC4, seed: 42);
// var (a4, v4) = mctsC4.Recherche(jeuC4.EtatInitial(), iterations: 2000);
// Console.WriteLine($"Connect-4 - MCTS(2000) : colonne={a4}, valeur={v4:F3}");
Console.WriteLine("Exercice Connect-4 : completez ConnectFour (Exercice 3) puis decommentez le tournoi.");

Exercice Connect-4 : completez ConnectFour (Exercice 3) puis decommentez le tournoi.


## Synthese

### Resume des approches

| Approche | Force | Faiblesse |
|----------|-------|----------|
| **Minimax + Alpha-Beta** | Optimal si arbre complet | Explosion combinatoire |
| **MCTS pur** | Pas de fonction d'evaluation | Convergence lente |
| **MCTS + Heuristiques** | Rapide, bon compromis | Heuristiques necessaires |
| **AlphaZero** | Auto-apprentissage | Ressources enormes |

### Quand utiliser quoi ?

- **Jeux simples** (Tic-Tac-Toe, petits puzzles) : Minimax
- **Jeux moyens** (Connect Four, Othello) : Alpha-Beta + heuristiques
- **Jeux complexes** (Go, Hex) : MCTS + reseaux de neurones
- **Jeux a information imparfaite** (Poker, Hanabi) : CFR + deep learning

### Pour aller plus loin

- **MuZero** : apprend les regles du jeu (model-based)
- **AlphaFold** : applications hors jeux (biologie)
- **Expert Iteration** : combinaison expert/apprenti

---

**Navigation** : [<< Recherche adversariale (C#)](Search-6-AdversarialSearch-Csharp.ipynb) | [Index](../README.md) | [Version Python](Search-7-MCTS-And-Beyond.ipynb) | [Dancing Links >>](Search-8-DancingLinks.ipynb)

### References academiques

- von Neumann, J. (1928). Zur Theorie der Gesellschaftsspiele. Mathematische Annalen 100:295-320.
- Kocsis, L. & Szepesvari, C. (2006). Bandit Based Monte-Carlo Planning. ECML 2006, LNCS 4212:282-293.
- Silver, D., Huang, A., Maddison, C.J. et al. (2016). Mastering the game of Go with deep neural networks and tree search. Nature 529:484-489.
- Silver, D., Schrittwieser, J., Simonyan, K. et al. (2017). Mastering the game of Go without human knowledge. Nature 550:354-359.
- Browne, C.B., Powley, E., Whitehouse, D. et al. (2012). A Survey of Monte Carlo Tree Search Methods. IEEE TCIAIG 4(1):1-43.
